# 04. Label the benchmark frames

**Goal:** give the benchmark its right answers. `bench.py` takes one frame every 30 seconds from each video, and here you mark every fish in them with one click each. Those dots are the **ground truth**: the labels we choose to trust.

Run these first, from `tidecount/server`:

```bash
uv run python ../ml/bench.py fetch --split dev
uv run python ../ml/bench.py frames --split dev
```

**Dev and locked videos.** Anyone can label and look at the `dev` videos. The `locked` videos stand in for Coastal Action's footage, so one person labels them, and nobody who tunes a model looks at them or their labels until the final score.

## 1. Choose a video

The table shows what's ready. Set `VIDEO` to an id that has frames.

In [ ]:
import bench
from labeler import FishLabeler

bench.status(bench.manifest())

In [ ]:
VIDEO = "holston-trout"

video = next(v for v in bench.manifest() if v.id == VIDEO)
frames = bench.label_order(video)
print(len(frames), "frames, in a fixed random order")

## 2. Mark the fish

- **Click the middle of each fish.** Include partly hidden, blurry or far-away fish if you're confident it's a fish.
- **Click a dot again** to remove it, or press **Z** to undo the last one.
- **No fish in the frame?** Press **N**. A frame with no dots counts as "no fish", and that answer is just as useful.
- **Too dark or blurry to tell?** Press **U**. That's different from "no fish", and those frames are left out of the score.
- **Count only fish for now.** Crabs and other animals get their own labels later.
- **Bait isn't a fish.** Two videos have a bait bag or basket in view. Don't mark what's inside it.

Each answer is saved straight away, so you can stop and come back. The frames are shuffled, so the ones you finish still cover the whole clip.

**You don't have to finish a video.** About 30 to 40 frames per video is enough for a first score, because the shuffled frames are a fair sample of the clip. Start with the videos that have few fish per frame, and leave `edmonds-crab-cam` for last.

Frames are shown at full size and scaled to fit the page. Zoom the browser in if a fish is too small to click.

In [ ]:
labeler = FishLabeler(frames, video.labels)
labeler

## 3. Score

When the dev videos are labelled, run the detectors and write the score sheet:

```bash
uv run python ../ml/bench.py detect --split dev
uv run python ../ml/bench.py score --split dev
```

The sheet is saved to `data/bench/scores-dev.md`. Scoring `dev` also freezes each setup's thresholds, and `score --split locked` reuses them, so no setting is ever chosen on the test videos.